In [0]:
# ---------------------------------------------------------
# 0. Set-up Imports
# ---------------------------------------------------------

import os
import uuid
import hashlib
import logging
import sys 
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable


In [0]:
# ---------------------------------------------------------
# 0.1 Standardize Column Names
# ---------------------------------------------------------

def standardize_column_names(df):

    standardized_columns = []

    for column in df.columns:

        standardized_column = (
            column
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("/", "_")
        )

        standardized_columns.append(
            standardized_column
        )

    return df.toDF(*standardized_columns)

In [0]:
# ---------------------------------------------------------
# 0.2 Enforce Schema onto DataFrame
# ---------------------------------------------------------

def apply_silver_schema(df, schema: StructType):
    for field in schema.fields:
        if field.name in df.columns:
            # Check if the target type is numeric
            is_numeric_type = isinstance(
                field.dataType, 
                (DecimalType, IntegerType, DoubleType, FloatType, LongType, ShortType, ByteType)
            )
            
            if is_numeric_type:
                # Clean commas and whitespace before casting numeric types
                df = df.withColumn(
                    field.name,
                    F.regexp_replace(F.col(field.name), ",", "").cast(field.dataType)
                )
            else:
                # Non-numeric types cast directly
                df = df.withColumn(
                    field.name,
                    F.col(field.name).cast(field.dataType)
                )
    
    return df

In [0]:
# ---------------------------------------------------------
# 0.3 Validated Columns for NULLs
# ---------------------------------------------------------

def validate_required_columns(
    df,
    required_columns
):

    validation_condition = None

    for column in required_columns:

        condition = F.col(column).isNull()

        if validation_condition is None:

            validation_condition = condition

        else:

            validation_condition = (
                validation_condition | condition
            )

    invalid_df = df.filter(validation_condition)

    valid_df = df.filter(~validation_condition)

    return valid_df, invalid_df

In [0]:
# ---------------------------------------------------------
# 0.4 Deduplicate Records
# ---------------------------------------------------------

def deduplicate_records(
    df,
    primary_key_columns,
    order_column
):

    window_spec = Window.partitionBy(
        *primary_key_columns
    ).orderBy(
        F.col(order_column).desc()
    )

    dedup_df = (
        df
        .withColumn(
            "row_num",
            F.row_number().over(window_spec)
        )
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    return dedup_df

In [0]:
# ---------------------------------------------------------
# 0.5 Apply Metadata Columns
# ---------------------------------------------------------

def apply_silver_metadata(df):

    return (
        df

        .withColumn(
            "silver_processed_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "silver_load_date",
            F.current_date()
        )
    )

In [0]:
# ---------------------------------------------------------
# 0.6 Rename Bronze Metadata Columns
# ---------------------------------------------------------

def rename_bronze_metadata(df):

    return (
         df
         
         .withColumnsRenamed({
        "bronze_file_hash": "source_file_hash",
        "bronze_ingestion_timestamp": "record_ingestion_timestamp",
        "bronze_batch_id": "batch_id"
})
    )

In [0]:
# ---------------------------------------------------------
# 0.7 Drop Excess Bronze Metadata Columns
# ---------------------------------------------------------

def drop_excess_bronze_metadata(df):

    return (
        df
        
        .drop(
        "bronze_source_file",
        "bronze_file_name",
        "bronze_pipeline_run_id")
    )

In [0]:
# ---------------------------------------------------------
# 0.8 Add Transformed Records to Silver Table
# ---------------------------------------------------------

def merge_to_silver(
    spark,
    df,
    silver_table,
    merge_condition
):

    if not spark.catalog.tableExists(silver_table):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )

        return

    delta_table = DeltaTable.forName(
        spark,
        silver_table
    )

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )